<a href="https://colab.research.google.com/github/ronan-cunha/curso-advanced-learning/blob/main/Notebooks/aula_12_random_forest_parametrizacao.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml, load_diabetes
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split

In [2]:
# ==============================================================================
# ETAPA 1: Carregar Dataset Real (Diabetes)
# ==============================================================================
# O dataset possui 442 observações e 10 variáveis preditoras clínicas:
# age (idade), sex (sexo), bmi (IMC), bp (pressão arterial média)
# s1 a s6 (seis medições do soro sanguíneo, ex: colesterol total, LDL, HDL, triglicérides, glicose)
# https://scikit-learn.org/stable/datasets/toy_dataset.html#diabetes-dataset
diabetes = load_diabetes(as_frame=True)
df = diabetes.frame
df.head()

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,151.0
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204,75.0
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930,141.0
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362,206.0
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641,135.0


In [11]:
# ==============================================================================
# ETAPA 2: Divisão entre Treino e Teste
# ==============================================================================
X = df.drop(columns=["target"])
y = df["target"]

# 1. Binarização do Target para Classificação
# - 1 se a progressão da doença estiver acima da mediana (alto risco).
# - 0 se estiver abaixo da mediana (baixo risco).
med_target = 100 #y.mean()
y_class = (y > med_target).astype(int)

feature_names = X.columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y_class, test_size=0.20, random_state=42
)

print(f"Amostras de Treino: {X_train.shape[0]} | Amostras de Teste: {X_test.shape[0]}\n")

Amostras de Treino: 353 | Amostras de Teste: 89



In [12]:
pd.concat([y_train.describe(), y_test.describe()], axis=1, keys=['Treino', 'Teste'])

,Treino,Teste
count,353.000000,89.000000
mean,0.677054,0.617978
std,0.468266,0.488635
min,0.000000,0.000000
25%,0.000000,0.000000
50%,1.000000,1.000000
75%,1.000000,1.000000
max,1.000000,1.000000


In [13]:
# ==============================================================================
# ETAPA 3: Otimização de Hiperparâmetros
# ==============================================================================

from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

# Definição do espaço de hiperparâmetros para busca
param_grid = {
    'n_estimators': [100, 500],
    'criterion': ['gini', 'entropy'],
    'max_features': [2, 3, 5, 8],
    'min_samples_leaf': [5, 10]
}

# Instanciando o modelo base
rf_base = RandomForestClassifier(random_state=42, n_jobs=-1)

In [16]:
# Configurando o Grid Search com 5-fold cross-validation
grid_search = GridSearchCV(
    estimator=rf_base,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

print("Iniciando Grid Search...")
grid_search.fit(X_train, y_train)

# Melhores parâmetros e resultado do Grid Search
best_rf_grid = grid_search.best_estimator_
y_pred_grid = best_rf_grid.predict(X_test)
acc_grid = accuracy_score(y_test, y_pred_grid)

print(f"\nMelhores Parâmetros (Grid Search): {grid_search.best_params_}")
print(f"Acurácia no Teste (Grid Search): {acc_grid:.4f}")

Iniciando Grid Search...
Fitting 5 folds for each of 32 candidates, totalling 160 fits

Melhores Parâmetros (Grid Search): {'criterion': 'gini', 'max_features': 2, 'min_samples_leaf': 10, 'n_estimators': 100}
Acurácia no Teste (Grid Search): 0.7191


In [17]:
# Configurando o Random Search (testando 30 combinações aleatórias)
random_search = RandomizedSearchCV(
    estimator=rf_base,
    param_distributions=param_grid,
    n_iter=30,
    cv=5,
    scoring='accuracy',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

print("Iniciando Random Search...")
random_search.fit(X_train, y_train)

# Melhores parâmetros e resultado do Random Search
best_rf_random = random_search.best_estimator_
y_pred_random = best_rf_random.predict(X_test)
acc_random = accuracy_score(y_test, y_pred_random)

print(f"\nMelhores Parâmetros (Random Search): {random_search.best_params_}")
print(f"Acurácia no Teste (Random Search): {acc_random:.4f}")

Iniciando Random Search...
Fitting 5 folds for each of 30 candidates, totalling 150 fits

Melhores Parâmetros (Random Search): {'n_estimators': 100, 'min_samples_leaf': 10, 'max_features': 2, 'criterion': 'gini'}
Acurácia no Teste (Random Search): 0.7191


In [10]:
print("--- Comparação de Desempenho dos Modelos ---")
print(f"Grid Search Tuning  - Acurácia: {acc_grid:.4f} | Erro: {1.0 - acc_grid:.4f}")
print(f"Random Search Tuning- Acurácia: {acc_random:.4f} | Erro: {1.0 - acc_random:.4f}")

--- Comparação de Desempenho dos Modelos ---
Grid Search Tuning  - Acurácia: 0.7079 | Erro: 0.2921
Random Search Tuning- Acurácia: 0.7079 | Erro: 0.2921
